# **SheriaLens**

## **Overview**
________________

SheriaLens is an AI-powered legal intelligence engine that aims to democratize access to justice. By leveraging Natural Language Processing (NLP) and Large Language Models (LLMs), we are building a computational layer for the legal system—transforming static, impenetrable legalese into clear, actionable insights.

Our objective is thus to build a RAG (Retrieval-Augmented Generation) pipeline that parses, indexes and synthesizes statutory data to answer legal queries with high fidelity.

## **Business Understanding**
______________________________

The law is open, but it is not accessible. "Legalese" acts as a cryptographic barrier, keeping the general public and SMEs in the dark while professional counsel remains prohibitively expensive.


We are not replacing lawyers; we are automating the discovery phase of legal work:
 * *For the Public:* Instant interpretation of rights and obligations.
 * *For Professionals:* High-speed retrieval of precedents and statutes.
 * *The Value:* Reducing the marginal cost of legal understanding to zero.

## **Data Understanding**
________________________

Our dataset is the bedrock of the judicial system. It consists of a high-dimensional, unstructured corpus of text including:
 1.  *The Constitution:* The root node of legal logic.
 2.  *Case Law:* Codified rules (Penal Code, Traffic Act, etc.).
 3.  *Case Law:* The interpretive layer of judicial precedence.


In [18]:
# Loading the necessary libraries
import pandas as pd
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()

True

### **Web Scraping the data**

To build a legal AI that is both authoritative and hallucination-resistant, we cannot rely on fragmented third-party summaries. We must go to the fountainhead. We are architecting a precision retrieval pipeline to ingest data directly from the eKLR (Kenya Law) repository, which is the official publisher of the Laws of Kenya. While eKLR has successfully digitized the nation’s legal archives, this data currently exists as static, unstructured text—readable by humans, but opaque to machines. 

By systematically extracting and structuring this public-sector information, SheriaLens is not merely "scraping" a website; we are operationalizing the constitutional right to information (Article 35). We are transforming the static "letter of the law" into a dynamic, machine-readable knowledge graph, effectively extending eKLR's mission of universal access by converting their repository into the fuel for the next generation of Access-to-Justice technology.

In [25]:
# Cropping the page to remove the headers and footers
import pdfplumber
import re
import IPython
from IPython.display import display
path = r"Datasets\Raw_data\constitution\TheConstitutionOfKenya.pdf"

def crop_constituition(page):
    width = page.width
    height = page.height
    crop_box = (0, height * 0.10, width, height * 0.90)
    cropped_page = page.crop(bbox=crop_box)
    return cropped_page

display(cropped_page)

<Page:17>

In [19]:
import pdfplumber
from IPython.display import display

# 1. Define your replacement dictionary at the top
replacements = {
    "\ue000": "ff",   # affadavit
    "\ue001": "fi",   # finding
    "\ue002": "fl",   # conflict
    "\ue003": "ffi",  # office
    "\ue004": "ffl",  # waffle
    "’": "'",         # Normalize curly quotes
    "“": '"',         # Normalize curly double quotes
    "”": '"'
}

path = r"Datasets\Raw_data\case_laws\KEELC\2025\February\keelc_2025_640.pdf"

with pdfplumber.open(path) as pdf:
    full_text = []
    
    for i, page in enumerate(pdf.pages):
        # Conditional cropping based on page number
        if i == 0:
            cropped_page = page.crop((0, 138, page.width, page.height - 65))
        else:
            cropped_page = page.crop((0, 0, page.width, page.height - 65))
        
        # (Optional) Image generation for checking crop accuracy
        # cropped_image = cropped_page.to_image(resolution=900) 
        
        # 2. Extract the text
        text = cropped_page.extract_text()
        
        # 3. Apply the replacements ONLY if text was found
        if text:
            for bad_char, good_char in replacements.items():
                text = text.replace(bad_char, good_char)
            
            full_text.append(text)

    # Print the final cleaned text
    print("\n".join(full_text))

Werimo (Suing as the Legal Representative of the Estate of Deceased
Werimo Otwila) v Okwomi & 6 others (Environment & Land Case
30 of 2015) [2025] KEELC 640 (KLR) (18 February 2025) (Judgment)
Neutral citation: [2025] KEELC 640 (KLR)
REPUBLIC OF KENYA
IN THE ENVIRONMENT AND LAND COURT AT BUSIA
ENVIRONMENT & LAND CASE 30 OF 2015
BN OLAO, J
FEBRUARY 18, 2025
BETWEEN
ROSETILA ATHIENO WERIMO (SUING AS THE LEGAL REPRESENTATIVE
OF THE ESTATE OF DECEASED WERIMO OTWILA) ....................... PLAINTIFF
AND
SUSY AYUMA OKWOMI ................................................................ 1ST DEFENDANT
LINUS PETER NAMBIRO ............................................................. 2ND DEFENDANT
MICHAEL ONYANGO RANGINYA ............................................ 3RD DEFENDANT
GEORGE WILLIAM WASIKEH ..................................................... 4TH DEFENDANT
TEDDY DUMSANE NDONDI ....................................................... 5TH DEFENDANT
COUNTY GOVERNMENT OF BUSIA .........